# 1. Project Introduction

## E-commerce Funnel & Revenue Intelligence Analysis

This notebook analyzes end-to-end e-commerce performance to identify funnel bottlenecks, revenue drivers, and strategic growth opportunities.

# 2. Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f"{x:,.2f}")

# 3. Data Loading

In [ ]:
BASE_DIR = Path.cwd()

customers = pd.read_csv(BASE_DIR / 'customers.csv')
products = pd.read_csv(BASE_DIR / 'products.csv')
orders = pd.read_csv(BASE_DIR / 'orders.csv')
order_items = pd.read_csv(BASE_DIR / 'order_items.csv')
events = pd.read_csv(BASE_DIR / 'events.csv')
sessions = pd.read_csv(BASE_DIR / 'sessions.csv')
reviews = pd.read_csv(BASE_DIR / 'reviews.csv')

datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
    'events': events,
    'sessions': sessions,
    'reviews': reviews,
}

for name, frame in datasets.items():
    print(f"{name:12} -> rows: {len(frame):>8,}, cols: {frame.shape[1]}")

# 4. Data Understanding

In [ ]:
summary_rows = []
for name, frame in datasets.items():
    summary_rows.append(
        {
            'dataset': name,
            'rows': frame.shape[0],
            'columns': frame.shape[1],
            'memory_mb': frame.memory_usage(deep=True).sum() / (1024 ** 2),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('rows', ascending=False)
summary_df

In [ ]:
for name, frame in datasets.items():
    print(f"
{name.upper()} COLUMNS")
    display(pd.DataFrame({'column': frame.columns, 'dtype': frame.dtypes.astype(str)}))

# 5. Data Quality Assessment

In [ ]:
quality_rows = []
for name, frame in datasets.items():
    quality_rows.append(
        {
            'dataset': name,
            'missing_values': int(frame.isna().sum().sum()),
            'duplicate_rows': int(frame.duplicated().sum()),
            'missing_pct': frame.isna().sum().sum() / (frame.shape[0] * frame.shape[1]) * 100,
        }
    )

quality_df = pd.DataFrame(quality_rows).sort_values('missing_values', ascending=False)
quality_df

In [ ]:
issue_summary = pd.DataFrame(
    [
        ['order_items duplicate rows', 'Inflated units/revenue and margin', 'Drop exact duplicates'],
        ['Event contextual nulls', 'False alarms if treated as generic data loss', 'Use event-type aware interpretation'],
        ['Datetime as object', 'Blocks trend and temporal analytics', 'Convert to datetime columns'],
    ],
    columns=['Issue', 'Impact', 'Resolution']
)
issue_summary

# 6. Data Cleaning

In [ ]:
# Keep copies for before-after comparison
order_items_before = len(order_items)

# Parse datetime fields
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
orders['order_time'] = pd.to_datetime(orders['order_time'])
events['timestamp'] = pd.to_datetime(events['timestamp'])
sessions['start_time'] = pd.to_datetime(sessions['start_time'])
reviews['review_time'] = pd.to_datetime(reviews['review_time'])

# Remove duplicates in order_items
order_items = order_items.drop_duplicates().copy()

# Standardize text fields
for frame, cols in [
    (orders, ['payment_method', 'country', 'device', 'source']),
    (sessions, ['device', 'source', 'country']),
    (products, ['category']),
]:
    for col in cols:
        frame[col] = frame[col].astype(str).str.strip()

# Basic validity checks
assert (orders['total_usd'] >= 0).all()
assert (orders['subtotal_usd'] >= orders['total_usd']).all()
assert (order_items['quantity'] > 0).all()

print('order_items rows before:', order_items_before)
print('order_items rows after :', len(order_items))
print('duplicates removed     :', order_items_before - len(order_items))

# 7. Feature Engineering

In [ ]:
orders['order_month'] = orders['order_time'].dt.to_period('M').astype(str)
orders['order_quarter'] = orders['order_time'].dt.to_period('Q').astype(str)
orders['order_weekday'] = orders['order_time'].dt.day_name()

order_items_ext = order_items.merge(
    products[['product_id', 'category', 'margin_usd']],
    on='product_id',
    how='left'
)
order_items_ext['margin_total'] = order_items_ext['quantity'] * order_items_ext['margin_usd']

customer_value = orders.groupby('customer_id', as_index=False)['total_usd'].sum()
customer_value['customer_value_segment'] = pd.qcut(
    customer_value['total_usd'],
    q=4,
    labels=['Low', 'Mid', 'High', 'VIP']
)

product_revenue = order_items_ext.groupby('product_id', as_index=False)['line_total_usd'].sum()
product_revenue = product_revenue.sort_values('line_total_usd', ascending=False)
product_revenue['cum_share'] = product_revenue['line_total_usd'].cumsum() / product_revenue['line_total_usd'].sum()
product_revenue['abc_class'] = pd.cut(
    product_revenue['cum_share'],
    bins=[0, 0.8, 0.95, 1.0],
    labels=['A', 'B', 'C'],
    include_lowest=True
)

customer_value.head()

# 8. EDA

## Univariate, Bivariate, and Multivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(orders['total_usd'], bins=40, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Order Value Distribution')
axes[0].set_xlabel('total_usd')

sns.boxplot(x=orders['total_usd'], ax=axes[1], color='darkorange')
axes[1].set_title('Order Value Boxplot')
axes[1].set_xlabel('total_usd')

plt.tight_layout()
plt.show()

In [ ]:
category_perf = order_items_ext.groupby('category', as_index=False).agg(
    revenue=('line_total_usd', 'sum'),
    units=('quantity', 'sum'),
    margin=('margin_total', 'sum')
).sort_values('revenue', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=category_perf, x='revenue', y='category', palette='viridis', ax=ax)
ax.set_title('Category Revenue Comparison')
ax.set_xlabel('Revenue (USD)')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
analysis_df = orders[['subtotal_usd', 'discount_pct', 'total_usd']].copy()
analysis_df['avg_item_qty'] = order_items.groupby('order_id')['quantity'].sum().reindex(orders['order_id']).values
corr = analysis_df.corr(numeric_only=True)

plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# 9. KPI Analysis

In [ ]:
kpis = {
    'Total Revenue (USD)': orders['total_usd'].sum(),
    'Total Orders': orders['order_id'].nunique(),
    'Unique Purchasing Customers': orders['customer_id'].nunique(),
    'Average Order Value (USD)': orders['total_usd'].mean(),
    'Repeat Purchase Rate': (orders['customer_id'].value_counts().gt(1).sum() / orders['customer_id'].nunique()),
    'Average Items per Order': order_items.groupby('order_id')['quantity'].sum().mean(),
    'Average Discount %': orders['discount_pct'].mean(),
    'Estimated Gross Margin %': order_items_ext['margin_total'].sum() / orders['total_usd'].sum(),
    'Average Rating': reviews['rating'].mean(),
}

kpi_df = pd.DataFrame({'KPI': list(kpis.keys()), 'Current Value': list(kpis.values())})
kpi_df

In [ ]:
funnel = events.groupby('event_type')['session_id'].nunique().reindex(
    ['page_view', 'add_to_cart', 'checkout', 'purchase']
).fillna(0).astype(int)

funnel_df = funnel.reset_index()
funnel_df.columns = ['stage', 'sessions']
funnel_df['session_rate'] = funnel_df['sessions'] / sessions['session_id'].nunique()
funnel_df

# 10. Business Analysis

In [ ]:
monthly = orders.groupby('order_month', as_index=False).agg(
    revenue=('total_usd', 'sum'),
    orders=('order_id', 'nunique'),
    customers=('customer_id', 'nunique')
)
monthly['revenue_growth_pct'] = monthly['revenue'].pct_change() * 100
monthly.tail()

In [ ]:
source_perf = orders.groupby('source', as_index=False).agg(
    revenue=('total_usd', 'sum'),
    orders=('order_id', 'nunique'),
    aov=('total_usd', 'mean')
).sort_values('revenue', ascending=False)

device_perf = orders.groupby('device', as_index=False).agg(
    revenue=('total_usd', 'sum'),
    orders=('order_id', 'nunique'),
    aov=('total_usd', 'mean')
).sort_values('revenue', ascending=False)

source_perf, device_perf

# 11. Visualization

In [ ]:
# Revenue trend
plt.figure(figsize=(14, 5))
plt.plot(monthly['order_month'], monthly['revenue'], marker='o', linewidth=2)
plt.xticks(rotation=75)
plt.title('Monthly Revenue Trend')
plt.xlabel('Order Month')
plt.ylabel('Revenue (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Funnel chart (bar)
plt.figure(figsize=(8, 5))
sns.barplot(data=funnel_df, x='stage', y='sessions', palette='Blues_d')
plt.title('Funnel Stage Sessions')
plt.xlabel('Funnel Stage')
plt.ylabel('Unique Sessions')
plt.tight_layout()
plt.show()

In [ ]:
# Top category contributors
top_categories = category_perf.head(7)
plt.figure(figsize=(10, 6))
sns.barplot(data=top_categories, y='category', x='revenue', palette='magma')
plt.title('Top Category Revenue Contributors')
plt.xlabel('Revenue (USD)')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
# KPI summary chart
plot_kpis = kpi_df.copy()
plot_kpis = plot_kpis[plot_kpis['KPI'].isin([
    'Total Revenue (USD)',
    'Total Orders',
    'Unique Purchasing Customers',
    'Average Order Value (USD)'
])]

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_kpis, x='KPI', y='Current Value', palette='Set2')
plt.title('Selected KPI Snapshot')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 12. Key Insights

1. Major drop-off appears between add-to-cart and checkout.
2. Mobile drives the largest order and revenue share.
3. Organic and direct channels are top scale contributors.
4. VIP segment contributes a disproportionate share of total revenue.
5. Category leadership is distributed, reducing single-category dependency risk.
6. Product concentration is moderate rather than extreme.
7. Repeat purchase behavior is strong and valuable.
8. Gross margin profile remains healthy.
9. Revenue trend is stable long-term but volatile month-to-month.
10. Review sentiment is broadly positive and can be leveraged for conversion uplift.

# 13. Recommendations

- Prioritize cart-to-checkout UX optimization to reduce abandonment.
- Scale high-quality traffic sources (organic/referral mix).
- Build VIP retention programs with proactive churn prevention.
- Use category scorecards to optimize promotional and inventory strategy.
- Operationalize review-driven merchandising on product and landing pages.

# 14. Conclusion

The business demonstrates strong revenue scale, healthy margin potential, and positive customer sentiment. The largest near-term growth opportunity is improving checkout initiation from cart, while long-term performance can be strengthened through segment-led retention, channel quality expansion, and category-level optimization.